In [2]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import glob
import seaborn as sns
import sys
import copy
from tqdm.notebook import tqdm
from numba import jit
from scipy import stats
import networkx as nx
import random
import re
from numba import njit

plt.style.use('seaborn-deep')
plt.rcParams["text.usetex"] = True
plt.rcParams['text.latex.preamble'] = r'\usepackage{amssymb,amsmath}'

plt.rcParams["figure.figsize"] = 11.7, 8.3
plt.rcParams["figure.dpi"] = 75

plt.rcParams["font.size"] = 24
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Fira Sans", 'PT Sans', 'Open Sans', 'Roboto', 'DejaVu Sans', 'Liberation Sans', 'sans-serif']

plt.rcParams["legend.frameon"] = True
plt.rcParams["legend.fancybox"] = True
plt.rcParams["legend.fontsize"] = "small"

plt.rcParams["lines.linewidth"] = 2.5
plt.rcParams["lines.markersize"] = 14
plt.rcParams["lines.markeredgewidth"] = 2

plt.rcParams["xtick.major.size"] = 8
plt.rcParams["ytick.major.size"] = 8

C:\Users\merit\AppData\Local\Temp\ipykernel_8084\4293947706.py:19: MatplotlibDeprecationWarning: The seaborn styles shipped by Matplotlib are deprecated since 3.6, as they no longer correspond to the styles shipped by seaborn. However, they will remain available as 'seaborn-v0_8-<style>'. Alternatively, directly use the seaborn API instead.
  plt.style.use('seaborn-deep')


# Taken from https://github.com/giotto-ai/giotto-tda/blob/master/examples/classifying_shapes.ipynb

In [3]:
from generate_datasets import make_point_clouds
num_points = 20
num_samples = 10
point_clouds_basic, labels_basic = make_point_clouds(n_samples_per_shape=num_samples, n_points=num_points, noise=0.5)
point_clouds_basic.shape, labels_basic.shape

((30, 400, 3), (30,))

In [4]:
from gtda.plotting import plot_point_cloud

plot_point_cloud(point_clouds_basic[0])

In [5]:
plot_point_cloud(point_clouds_basic[10])

In [6]:
plot_point_cloud(point_clouds_basic[-1])

In [10]:
from pynndescent import NNDescent
import math

def euclidean_distance_3d(p, q):
    return math.sqrt((p[0] - q[0])**2 + (p[1] - q[1])**2 + (p[2] - q[2])**2)

# graph construction using knn
k_values = [4, 5, 6, 7, 8, 9, 10, 20, 30]

for num_cloud in range(num_samples):
    for k in k_values:
        
        point_cloud = point_clouds_basic[num_cloud] # We can then change the dataset

        # Compute kNN using NNDescent
        # indices: array where each row contains the indices of the k-nearest neighbors for each point.
        # I put k+1 because the algorithm counts the same node as the nearest
        knnData = NNDescent(point_cloud, metric="euclidean", n_neighbors=k+1, verbose=True)
        indices, distances = knnData.neighbor_graph

        # Create graph
        G = nx.Graph()

        for i in range(len(point_clouds_basic[0])):
            G.add_node(i, pos=tuple(point_cloud[i]))

        # Add edges based on kNN
        for i in range(len(point_cloud)):
            for j in indices[i]:  
                if i!=j: # this is to exclude self loops
                    G.add_edge(i, j)

        # Save the graph to an edgelist file
        output_filename = f"graphs/pointcloud_{num_cloud}_knn_{k}.edgelist"
        nx.write_edgelist(G, output_filename, data=False) 

        print(f"Graph saved to {output_filename}")

# graph construction using rgg
r_values = [0.1, 0.3, 0.5, 0.7, 0.9, 1.1, 1.3, 1.5]

for num_cloud in range(num_samples):
    for r in r_values:
        
        point_cloud = point_clouds_basic[num_cloud] 

        G = nx.Graph()

        for i in range(len(point_clouds_basic[0])):
            G.add_node(i, pos=tuple(point_cloud[i]))

        # Add edges based on random geometric graph
        for i in range(len(point_cloud)):
            for j in range(len(point_cloud)):  
                if i!=j:
                    if euclidean_distance_3d(point_cloud[i],point_cloud[j]) < r:
                        G.add_edge(i, j)

        output_filename = f"graphs/pointcloud_{num_cloud}_rgg_{r}.edgelist"
        nx.write_edgelist(G, output_filename, data=False) 

        print(f"Graph saved to {output_filename}")

Thu Jan 30 22:09:19 2025 Building RP forest with 9 trees
Thu Jan 30 22:09:19 2025 NN descent for 9 iterations
	 1  /  9
	 2  /  9
	Stopping threshold met -- exiting after 2 iterations
Graph saved to graphs/pointcloud_0_knn_4.edgelist
Thu Jan 30 22:09:19 2025 Building RP forest with 9 trees
Thu Jan 30 22:09:19 2025 NN descent for 9 iterations
	 1  /  9
	 2  /  9
	Stopping threshold met -- exiting after 2 iterations
Graph saved to graphs/pointcloud_0_knn_5.edgelist
Thu Jan 30 22:09:19 2025 Building RP forest with 9 trees
Thu Jan 30 22:09:19 2025 NN descent for 9 iterations
	 1  /  9
	 2  /  9
	 3  /  9
	Stopping threshold met -- exiting after 3 iterations
Graph saved to graphs/pointcloud_0_knn_6.edgelist
Thu Jan 30 22:09:19 2025 Building RP forest with 9 trees
Thu Jan 30 22:09:19 2025 NN descent for 9 iterations
	 1  /  9
	 2  /  9
	Stopping threshold met -- exiting after 2 iterations
Graph saved to graphs/pointcloud_0_knn_7.edgelist
Thu Jan 30 22:09:19 2025 Building RP forest with 9 tre

In [19]:
from filtrations import density_sum_cycles
from diagrams import calculate_persistence_diagram

filename = "graphs/pointcloud_0_knn_4.edgelist"
N = nx.read_edgelist(filename, nodetype=int)  

k = 1  # change k later
persistence_diagram = calculate_persistence_diagram(N, density_sum_cycles, k)
print(persistence_diagram)
# basically the current problem is that our filtration returns an empty array

hola
hola2
filtration valuees: []


KeyError: 'filtration'

In [21]:
from gtda.homology import VietorisRipsPersistence

# Track connected components, loops, and voids
homology_dimensions = [0, 1, 2]

# Collapse edges to speed up H2 persistence calculation!
persistence = VietorisRipsPersistence(
    metric="euclidean",
    homology_dimensions=homology_dimensions,
    n_jobs=6,
    collapse_edges=True,
)

diagrams_basic = persistence.fit_transform(point_clouds_basic)

In [7]:
from gtda.plotting import plot_diagram

plot_diagram(diagrams_basic[0])

In [8]:
plot_diagram(diagrams_basic[10])

In [9]:
plot_diagram(diagrams_basic[-1])

In [10]:
from gtda.diagrams import PersistenceEntropy

persistence_entropy = PersistenceEntropy()

# calculate topological feature matrix
X_basic = persistence_entropy.fit_transform(diagrams_basic)

# expect shape - (n_point_clouds, n_homology_dims)
X_basic.shape

(30, 3)

In [11]:
plot_point_cloud(X_basic)

In [12]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(oob_score=True)
rf.fit(X_basic, labels_basic)

print(f"OOB score: {rf.oob_score_:.3f}")

OOB score: 1.000


In [13]:
from gtda.pipeline import Pipeline

steps = [
    ("persistence", VietorisRipsPersistence(metric="euclidean", homology_dimensions=homology_dimensions, n_jobs=6)),
    ("entropy", PersistenceEntropy()),
    ("model", RandomForestClassifier(oob_score=True)),
]

pipeline = Pipeline(steps)

In [14]:
pipeline.fit(point_clouds_basic, labels_basic)

Pipeline(steps=[('persistence',
                 VietorisRipsPersistence(homology_dimensions=[0, 1, 2],
                                         n_jobs=6)),
                ('entropy', PersistenceEntropy()),
                ('model', RandomForestClassifier(oob_score=True))])

In [15]:
pipeline["model"].oob_score_

1.0

---
---

In [16]:
from openml.datasets.functions import get_dataset

df = get_dataset('shapes').get_data(dataset_format='dataframe')[0]
df.head()

,x,y,z,target
0,0.341007,0.318606,0.096725,human_arms_out9
1,0.329226,0.421601,0.056749,human_arms_out9
2,0.446869,0.648674,0.124090,human_arms_out9
3,0.314729,0.217860,0.070847,human_arms_out9
4,0.426678,0.919195,0.047609,human_arms_out9


In [17]:
import openml

In [18]:
plot_point_cloud(df.query('target == "biplane0"')[["x", "y", "z"]].values)

In [19]:
import numpy as np

point_clouds = np.asarray(
    [
        df.query("target == @shape")[["x", "y", "z"]].values
        for shape in df["target"].unique()
    ]
)
point_clouds.shape

(40, 400, 3)

In [20]:
persistence = VietorisRipsPersistence(
    metric="euclidean",
    homology_dimensions=homology_dimensions,
    n_jobs=6,
    collapse_edges=True,
)
persistence_diagrams = persistence.fit_transform(point_clouds)

In [21]:
# Index - (human_arms_out, 0), (vase, 10), (dining_chair, 20), (biplane, 30)
index = 30
plot_diagram(persistence_diagrams[index])

In [22]:
persistence_entropy = PersistenceEntropy(normalize=True)
# Calculate topological feature matrix
X = persistence_entropy.fit_transform(persistence_diagrams)
# Visualise feature matrix
plot_point_cloud(X)

In [23]:
labels = np.zeros(40)
labels[10:20] = 1
labels[20:30] = 2
labels[30:] = 3

rf = RandomForestClassifier(oob_score=True, random_state=42)
rf.fit(X, labels)
rf.oob_score_

0.6